In [1]:
import io
import os
import random
import sys
import zipfile

import torch
import torchvision
from PIL import Image
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

assert sys.version_info >= (3, 10), "Python 3.10+ required"
print("Python     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Python     : 3.14.5
torch      : 2.14.0
torchvision: 0.29.0
Using mps device


# Sample Data

In [2]:
REPO = "rootstrap-org/waste-classifier"
LABEL_MAP = {  # dataset label -> our class name
    "paper": "paper",
    "plastic": "plastic",
    "glass": "glass",
    "compost": "compost",
    "trash": "landfill",
}
N_TRAIN = 10  # 10 images x 5 classes = 50 images total
IMG_EXTS = (".jpg", ".jpeg", ".png")

sample_dir = os.path.join("data", "waste_sample")
train_dir = os.path.join(sample_dir, "train")


def already_downloaded():
    return all(
        os.path.isdir(os.path.join(train_dir, c))
        and len(os.listdir(os.path.join(train_dir, c))) >= N_TRAIN
        for c in LABEL_MAP.values()
    )


def save_member(zf, member, out_path):
    """Read one image out of the zip, shrink it (some are 8k px wide) and save as JPEG."""
    with zf.open(member) as f:
        img = Image.open(io.BytesIO(f.read())).convert("RGB")
    img.thumbnail((512, 512))
    img.save(out_path, quality=90)


def sample_from_zip(zf, seed=0):
    rng = random.Random(seed)
    by_label = {}
    for name in zf.namelist():  # reads only the zip's central directory
        parts = [p for p in name.split("/") if p]
        if (
            name.endswith("/")
            or len(parts) < 2
            or "__MACOSX" in parts
            or parts[-1].startswith(".")
        ):
            continue
        if not parts[-1].lower().endswith(IMG_EXTS):
            continue
        label = parts[-2].lower()  # class = parent folder name
        if label in LABEL_MAP:
            by_label.setdefault(label, []).append(name)

    missing = [l for l in LABEL_MAP if l not in by_label]
    if missing:
        raise RuntimeError(
            f"Labels {missing} not found in zip. First entries: {zf.namelist()[:8]}"
        )

    for label, names in by_label.items():
        cls = LABEL_MAP[label]
        # prefer the dataset's own train split when the zip has one
        pool = [n for n in names if "train" in n.lower().split("/")] or names
        chosen = rng.sample(pool, N_TRAIN)
        out_dir = os.path.join(train_dir, cls)
        os.makedirs(out_dir, exist_ok=True)
        for i, m in enumerate(chosen):
            save_member(zf, m, os.path.join(out_dir, f"{cls}_{i:02d}.jpg"))
        print(f"  {label} -> {cls}: {N_TRAIN} images")


if already_downloaded():
    print("Sample already downloaded, skipping:", os.path.abspath(sample_dir))
else:
    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    zips = fs.glob(f"datasets/{REPO}/**/*.zip") or [
        p for p in fs.ls(f"datasets/{REPO}", detail=False) if p.endswith(".zip")
    ]
    if not zips:
        raise RuntimeError(
            "No .zip found in the dataset repo. Check https://huggingface.co/datasets/"
            + REPO
            + "/tree/main"
        )
    print("Reading zip:", zips[0])
    with fs.open(zips[0], "rb") as fh, zipfile.ZipFile(fh) as zf:
        sample_from_zip(zf)
    print("Saved to", os.path.abspath(sample_dir))

Sample already downloaded, skipping: /Users/kathleenmaung/Sortify/notebooks/data/waste_sample


# B. Torchvision ImageFolder Dataloader with Resize and ToTensor Transforms

In [7]:
data_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        ),  # ImageNet stats, as in the tutorial
    ]
)

image_dataset = datasets.ImageFolder(train_dir, data_transform)
dataloader = DataLoader(image_dataset, batch_size=4, shuffle=True)
class_names = image_dataset.classes
print(class_names, len(image_dataset), "images")
assert len(class_names) == 5, f"expected 5 classes, found {class_names}"

['compost', 'glass', 'landfill', 'paper', 'plastic'] 50 images


# C. Load ResNet-18, replace fc layer with 5 output units.


In [4]:
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Verification: output shape is (batch, 5)
x, y = next(iter(dataloader))
with torch.no_grad():
    out = model(x.to(device))
print("Output shape:", tuple(out.shape))
assert out.shape[1] == 5

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/kathleenmaung/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 24.3MB/s]


Output shape: (4, 5)


# D. Execute training loop for 2 epochs, printing batch loss every 5 steps

In [5]:
with torch.no_grad():
    initial_loss = criterion(out, y.to(device)).item()
print(f"Initial loss (before training): {initial_loss:.4f}")

epoch_losses = []
model.train()
for epoch in range(2):
    print(f"Epoch {epoch + 1}/2")
    running_loss = 0.0
    for step, (inputs, labels) in enumerate(dataloader, start=1):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        if step % 5 == 0:
            print(f"  step {step} | batch loss: {loss.item():.4f}")

    epoch_losses.append(running_loss / len(image_dataset))
    print(f"Epoch {epoch + 1} average loss: {epoch_losses[-1]:.4f}")

print(
    f"\nLoss went from {initial_loss:.4f} (start) to {epoch_losses[-1]:.4f} (end of epoch 2)"
)

Initial loss (before training): 1.7266
Epoch 1/2
  step 5 | batch loss: 1.9193
  step 10 | batch loss: 1.9019
Epoch 1 average loss: 1.7039
Epoch 2/2
  step 5 | batch loss: 1.7207
  step 10 | batch loss: 0.5732
Epoch 2 average loss: 1.1766

Loss went from 1.7266 (start) to 1.1766 (end of epoch 2)


# E. Pass an arbitrary test image through torch.softmax(model(img), dim=1) and print top category name

In [6]:
test_img_path, true_idx = random.choice(image_dataset.samples)
true_class = class_names[true_idx]

model.eval()
img = data_transform(Image.open(test_img_path).convert("RGB")).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(img), dim=1)

assert abs(probs.sum().item() - 1.0) < 1e-4

conf, pred = probs.max(dim=1)
print(f"Test image: {test_img_path}  (true class: {true_class})")
print({name: round(p, 4) for name, p in zip(class_names, probs[0].tolist())})
print(f"Predicted class: {class_names[pred.item()]} | confidence: {conf.item():.4f}")

Test image: data/waste_sample/train/glass/glass_08.jpg  (true class: glass)
{'compost': 0.0252, 'glass': 0.749, 'landfill': 0.0467, 'paper': 0.0365, 'plastic': 0.1426}
Predicted class: glass | confidence: 0.7490
